In [ ]:
import pandas as pd
import numpy as np

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

target1 = "Genetic Disorder"
target2 = "Disorder Subclass"

def clean_dataset(df, is_train=True):
    df = df.copy()

    df["missing_count"] = (df == -99).sum(axis=1)

    df = df.replace(-99, np.nan)
    df = df.replace(["Not Applicable", "Not applicable", "NA", "N/A","-","No record","Not available"], np.nan)

    drop_cols = [
        "Patient Id",
        "Patient First Name",
        "Family Name",
        "Father's name",
        "Institute Name",
        "Location of Institute",
        "Parental consent",
        "Place of birth"
    ]

    df = df.drop(columns=drop_cols, errors='ignore')

    missing_ratio = df.isnull().mean()
    cols_to_drop = missing_ratio[missing_ratio > 0.6].index
    df = df.drop(columns=cols_to_drop, errors='ignore')

    symptom_cols = [col for col in df.columns if "Symptom" in col]

    if len(symptom_cols) > 0:
        df[symptom_cols] = df[symptom_cols].fillna(0)
        df[symptom_cols] = df[symptom_cols].astype(float)

        df["symptomCount"] = df[symptom_cols].sum(axis=1)
        df["symptomMean"] = df[symptom_cols].mean(axis=1)
        df["symptomMax"] = df[symptom_cols].max(axis=1)
        df["symptomStd"] = df[symptom_cols].std(axis=1)
        df["symptomMin"] = df[symptom_cols].min(axis=1)
        df["symptomRange"] = df["symptomMax"] - df["symptomMin"]

    num_cols = df.select_dtypes(include=["int64", "float64"]).columns
    cat_cols = df.select_dtypes(include=["object"]).columns

    for col in num_cols:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)

    for col in cat_cols:
        freq = df[col].value_counts(normalize=True)
        rare = freq[freq < 0.01].index
        df[col] = df[col].replace(rare, "Other")

        mode_val = df[col].mode()[0] if not df[col].mode().empty else "Unknown"
        df[col] = df[col].fillna(mode_val)

    df = df.drop_duplicates()

    return df

train_clean = clean_dataset(train, is_train=True)
test_clean = clean_dataset(test, is_train=False)

common_cols = [col for col in train_clean.columns if col in test_clean.columns]

train_clean = train_clean[common_cols + [target1, target2]]
test_clean = test_clean[common_cols]

train_clean.to_csv("train_clean(1).csv", index=False)
test_clean.to_csv("test_clean(1).csv", index=False)
